# Block transplant (exploratory)
Reads `results/transplant/tx4a_transplants.csv` and `tx4a_partial.csv` (`zoo.transplant --collect`, `zoo.transplant_analysis`). Figure only.

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
t = pd.read_csv(ROOT / "results/transplant/tx4a_transplants.csv")
pt = pd.read_csv(ROOT / "results/transplant/tx4a_partial.csv")
COL = {"ct01x": "#2a78d6", "ct08x": "#eb6834", "ct64x": "#1baf7a"}
INK, MUTED, GRID = "#2b2b2a", "#8a8980", "#e4e3df"

In [ ]:
full = t[t.alpha == 1]
conds = [("restore: active h.7 into CT", dict(direction="restore", donor="active", part="h7"), "recipient"),
         ("restore: active h.4 into CT", dict(direction="restore", donor="active", part="h4"), "recipient"),
         ("control: naive h.7 into CT", dict(direction="restore", donor="naive", part="h7"), "recipient"),
         ("reverse: CT h.7 into active", dict(direction="reverse", part="h7"), "donor"),
         ("reverse: CT h.4 into active", dict(direction="reverse", part="h4"), "donor")]
fig, (a, b) = plt.subplots(1, 2, figsize=(11, 4.2), gridspec_kw={"width_ratios": [1.35, 1]})
for i, (lab, sel, dcol) in enumerate(conds):
    g = full.loc[np.logical_and.reduce([full[c] == v for c, v in sel.items()])]
    for j, d in enumerate(["ct01x", "ct08x", "ct64x"]):
        v = g[g[dcol] == d].f
        a.scatter(np.full(len(v), i + (j - 1) * 0.22) + np.linspace(-0.05, 0.05, len(v)), v, s=22, color=COL[d],
                  edgecolor="white", linewidth=0.8, zorder=3, label=d.replace("ct", "CT ") if i == 0 else None)
a.axhline(0.5, color=MUTED, lw=1, ls="--"); a.text(4.45, 0.51, "support threshold 0.5", ha="right", va="bottom", fontsize=7.5, color=MUTED)
a.axhline(0.2, color=MUTED, lw=0.8, ls=":"); a.text(4.45, 0.21, "against: < 0.2", ha="right", va="bottom", fontsize=7.5, color=MUTED)
a.axhline(0, color=INK, lw=0.6)
a.set_xticks(range(len(conds)), [c[0].replace(": ", ":\n") for c in conds], fontsize=7.5)
a.set_ylabel("recovered fraction f of the target_logprob gap"); a.set_ylim(-0.4, 0.6)
a.set_title("Full transplants, 5 seeds per dose (CT = counter-trained)", fontsize=9, loc="left")
a.legend(frameon=False, fontsize=8, loc="upper left", ncol=3)
alphas = [0, 0.25, 0.5, 0.75, 1]
for _, r in pt.iterrows():
    b.plot(alphas, [r.f_a0] + [r[f"f_a{x:g}"] for x in alphas[1:]], color=COL[r.model], lw=1.2, alpha=0.85)
for d in COL:
    b.plot([], [], color=COL[d], label=d.replace("ct", "CT "))
b.set_xlabel("a: final block = (1 - a) CT + a active"); b.set_ylabel("recovered fraction f")
b.set_title("Partial transplants: target_rate never above 0.003", fontsize=9, loc="left")
b.legend(frameon=False, fontsize=8)
for ax in (a, b):
    ax.spines[["top", "right"]].set_visible(False); ax.grid(axis="y", color=GRID, lw=0.6); ax.set_axisbelow(True)
fig.tight_layout()
fig.savefig(ROOT / "reports/figures/transplant.png", dpi=150, bbox_inches="tight")